# Frost API

## Sett opp API-nøkkel

In [1]:
CLIENT_ID = "0fa358f8-1ba2-4c05-bff7-2977868375c2"

## Definer Breive-koordinater

In [2]:
breive_coords = (59.57624, 7.28038)

## Hent alle stasjoner og filtrer på koordinater

In [3]:
import requests
import pandas as pd
from geopy.distance import geodesic

# Hent alle stasjoner fra Frost
url_stations = "https://frost.met.no/sources/v0.jsonld"
response = requests.get(url_stations, auth=(CLIENT_ID,''))
stations = response.json()['data']

# Lag liste med stasjoner som har koordinater, og regn avstand til Breive
stations_list = []
for s in stations:
    if 'geometry' in s and 'coordinates' in s['geometry']:
        coords = (s['geometry']['coordinates'][1], s['geometry']['coordinates'][0])  # [lat, lon]
        distance = geodesic(breive_coords, coords).km
        stations_list.append({
            'id': s['id'],
            'name': s['name'],
            'distance_km': distance
        })

stations_df = pd.DataFrame(stations_list)
stations_df = stations_df.sort_values('distance_km')
stations_df.head(5)  # viser de 5 nærmeste stasjonene



,id,name,distance_km
1001,SN40880,HOVDEN - LUNDANE,6.174906
1717,SN40840,RV9 HOVDEN II,7.509339
227,SN40890,BREIDVATN,9.569694
22,SN40905,RV9 BJÅEN,12.209966
1994,SN46390,HOLMEVATN,13.595188


In [4]:
# Velg nærmeste stasjon
nearest_station = stations_df.iloc[0]['id']
print("Henter data fra stasjon:", nearest_station)

Henter data fra stasjon: SN40880


## Lag funksjon for å hente historisk temperatur

In [5]:
def fetch_temperature(stasjon_id, start, end):
    url = f"https://frost.met.no/observations/v0.jsonld?sources={stasjon_id}&elements=air_temperature&referencetime={start}/{end}"
    response = requests.get(url, auth=(CLIENT_ID, ''))
    
    if response.status_code != 200:
        print("Feil:", response.status_code)
        return pd.DataFrame()
    
    data = response.json()
    observations = data.get('data', [])
    
    records = []
    for obs in observations:
        time = obs.get('referenceTime')
        value = obs.get('observations', [{}])[0].get('value')
        records.append({"timestamp": time, "temperature": value})
    
    return pd.DataFrame(records)

## Sett ønsket periode og velg stasjon

In [6]:
vinter_periods = [
    ("2024-11-01T00:00:00Z", "2024-11-30T23:00:00Z"),
    ("2024-12-01T00:00:00Z", "2024-12-31T23:00:00Z"),
    ("2025-01-01T00:00:00Z", "2025-01-31T23:00:00Z"),
    ("2025-11-01T00:00:00Z", "2025-11-30T23:00:00Z"),
    ("2025-12-01T00:00:00Z", "2025-12-31T23:00:00Z"),
    ("2026-01-01T00:00:00Z", "2026-01-20T22:00:00Z")
]

Hent data fra Frost API

In [7]:
def fetch_temperature(stasjon_id, start, end):
    url = f"https://frost.met.no/observations/v0.jsonld?sources={stasjon_id}&elements=air_temperature&referencetime={start}/{end}"
    response = requests.get(url, auth=(CLIENT_ID, ''))
    
    if response.status_code != 200:
        print("Feil:", response.status_code)
        return pd.DataFrame()
    
    data = response.json()
    observations = data.get('data', [])
    
    records = []
    for obs in observations:
        time = obs.get('referenceTime')
        value = obs.get('observations', [{}])[0].get('value')
        records.append({"timestamp": time, "temperature": value})
    
    return pd.DataFrame(records)

Hent vinterperioder

In [8]:
def fetch_temperature_periods_hourly(stasjon_id, periods):
    dfs = []
    for start, end in periods:
        df = fetch_temperature(stasjon_id, start, end)
        dfs.append(df)
    if dfs:
        weather_df = pd.concat(dfs, ignore_index=True)
        # Konverter til datetime og behold kun hele timer
        weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
        weather_df_hourly = weather_df[weather_df['timestamp'].dt.minute == 0].copy()
        weather_df_hourly = weather_df_hourly.sort_values('timestamp').reset_index(drop=True)
        return weather_df_hourly
    else:
        return pd.DataFrame()

Hent data fra vintermånedene

In [9]:
weather_df_hourly = fetch_temperature_periods_hourly(nearest_station, vinter_periods)
weather_df_hourly.head()

,timestamp,temperature
0,2024-11-01 00:00:00+00:00,7.9
1,2024-11-01 01:00:00+00:00,7.6
2,2024-11-01 02:00:00+00:00,7.6
3,2024-11-01 03:00:00+00:00,5.9
4,2024-11-01 04:00:00+00:00,4.5


## Tidsformat

In [10]:
# Konverter til datetime hvis ikke gjort
weather_df_hourly['timestamp'] = pd.to_datetime(weather_df_hourly['timestamp'])

# Formater som '2025-01-18 09:00:00.0000000'
weather_df_hourly['timestamp'] = weather_df_hourly['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S.%f0')

weather_df_hourly.head()

,timestamp,temperature
0,2024-11-01 00:00:00.0000000,7.9
1,2024-11-01 01:00:00.0000000,7.6
2,2024-11-01 02:00:00.0000000,7.6
3,2024-11-01 03:00:00.0000000,5.9
4,2024-11-01 04:00:00.0000000,4.5


## Lagre data som CSV

In [11]:
weather_df_hourly.to_csv("../../data/processed/Breive_winter_hourly_weather.csv", index=False)
print("CSV lagret med timebasert temperaturdata for vintermånedene!")

CSV lagret med timebasert temperaturdata for vintermånedene!
